# Setup e instalación


In [1]:
!pip install nnunetv2 nibabel scipy pandas tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 20.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 113.0 MB/s eta 0:00:0

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!nvidia-smi

Sun May  3 21:42:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   45C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Configurar rutas

In [4]:
import os
from pathlib import Path

DRIVE_BASE     = '/content/drive/MyDrive/VerSe_2020_Dataset/preprocessed_verse_for_training'
NNUNET_RAW     = '/content/nnunet_verse/nnUNet_raw'
NNUNET_PREPROC = '/content/nnunet_verse/nnUNet_preprocessed'
NNUNET_RESULTS = '/content/nnunet_verse/nnUNet_results'
DATASET_ID     = 507
DATASET_NAME   = f'Dataset{DATASET_ID:03d}_VerSe2020'
CONFIG         = '3d_lowres'
TRAINER        = 'nnUNetTrainer_10epochs'

for d in [NNUNET_RAW, NNUNET_PREPROC, NNUNET_RESULTS]:
    os.makedirs(d, exist_ok=True)

os.environ['nnUNet_raw']          = NNUNET_RAW
os.environ['nnUNet_preprocessed'] = NNUNET_PREPROC
os.environ['nnUNet_results']      = NNUNET_RESULTS

print('✓ Rutas configuradas')
print(f'  Config: {CONFIG}')
print(f'  Trainer: {TRAINER}')

✓ Rutas configuradas
  Config: 3d_lowres
  Trainer: nnUNetTrainer_10epochs


# Restaurar 3D lowres desde Drive

In [5]:
import subprocess
import shutil
from pathlib import Path
from tqdm.notebook import tqdm
import threading

!apt-get install -y rsync -q

def copy_with_rsync(src, dst, desc='Copiando'):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)

    all_files   = [f for f in src.rglob('*') if f.is_file()]
    total_files = len(all_files)
    total_size  = sum(f.stat().st_size for f in all_files)
    total_gb    = round(total_size / 1e9, 2)
    print(f'{desc}: {total_files} archivos ({total_gb} GB)')

    stop_flag = [False]

    def update_progress(pbar):
        while not stop_flag[0]:
            current = len(list(dst.rglob('*')))
            pbar.n = min(current, total_files)
            pbar.refresh()
            threading.Event().wait(1.0)

    with tqdm(total=total_files, unit='archivos', desc=desc,
              bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]') as pbar:
        t = threading.Thread(target=update_progress, args=(pbar,))
        t.start()
        cmd = [
            'rsync', '-r', '--info=progress2',
            '--no-perms', '--no-owner', '--no-group',
            '--inplace', '--whole-file',
            str(src) + '/', str(dst) + '/'
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        stop_flag[0] = True
        t.join()
        pbar.n = total_files
        pbar.refresh()

    if result.returncode == 0:
        print(f'✓ {desc} completado\n')
    else:
        print(f'✗ Error: {result.stderr[:200]}')

print('\nRestaurando 3D lowres desde Drive...\n')

copy_with_rsync(
    Path(DRIVE_BASE) / 'nnUNet_preprocessed' / DATASET_NAME / 'nnUNetPlans_3d_lowres',
    Path(NNUNET_PREPROC) / DATASET_NAME / 'nnUNetPlans_3d_lowres',
    'nnUNetPlans_3d_lowres'
)

copy_with_rsync(
    Path(DRIVE_BASE) / 'nnUNet_preprocessed' / DATASET_NAME / 'gt_segmentations',
    Path(NNUNET_PREPROC) / DATASET_NAME / 'gt_segmentations',
    'gt_segmentations'
)

print('Copiando JSONs...')
for json_file in ['nnUNetPlans.json', 'dataset.json', 'dataset_fingerprint.json']:
    src = Path(DRIVE_BASE) / 'nnUNet_preprocessed' / DATASET_NAME / json_file
    dst = Path(NNUNET_PREPROC) / DATASET_NAME / json_file
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists():
        shutil.copy2(str(src), str(dst))
        print(f'  ✓ {json_file}')

for json_file in ['splits_final.json']:
    src = Path(DRIVE_BASE) / 'nnUNet_preprocessed' / DATASET_NAME / json_file
    dst = Path(NNUNET_PREPROC) / DATASET_NAME / json_file
    if src.exists():
        shutil.copy2(str(src), str(dst))
        print(f'  ✓ {json_file}')

for json_file in ['dataset.json', 'splits_info.json']:
    src = Path(DRIVE_BASE) / 'nnUNet_raw' / DATASET_NAME / json_file
    dst = Path(NNUNET_RAW) / DATASET_NAME / json_file
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    if src.exists():
        shutil.copy2(str(src), str(dst))
        print(f'  ✓ {json_file}')

n_lowres = len(list((Path(NNUNET_PREPROC) / DATASET_NAME / 'nnUNetPlans_3d_lowres').glob('*')))
print(f'\n✓ nnUNetPlans_3d_lowres: {n_lowres} archivos')
print('✓ Restauración completada')

Reading package lists...
Building dependency tree...
Reading state information...
rsync is already the newest version (3.2.7-0ubuntu0.22.04.4).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.

Restaurando 3D lowres desde Drive...

nnUNetPlans_3d_lowres: 783 archivos (4.42 GB)


nnUNetPlans_3d_lowres:   0%|          | 0/783 [00:00<?, ?archivos/s]

✓ nnUNetPlans_3d_lowres completado

gt_segmentations: 261 archivos (0.15 GB)


gt_segmentations:   0%|          | 0/261 [00:00<?, ?archivos/s]

✓ gt_segmentations completado

Copiando JSONs...
  ✓ nnUNetPlans.json
  ✓ dataset.json
  ✓ dataset_fingerprint.json
  ✓ dataset.json
  ✓ splits_info.json

✓ nnUNetPlans_3d_lowres: 783 archivos
✓ Restauración completada


# Instalar trainer 10 epochs de nnUn-Net

In [6]:
import nnunetv2
from pathlib import Path

nnunet_dir  = Path(nnunetv2.__file__).parent
trainer_dir = nnunet_dir / 'training' / 'nnUNetTrainer' / 'variants' / 'training_length'
trainer_dir.mkdir(parents=True, exist_ok=True)

trainer_code = '''
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer

class nnUNetTrainer_10epochs(nnUNetTrainer):
    """Trainer de 10 epochs para benchmark de optimización"""
    max_num_epochs = 10
'''

trainer_path = trainer_dir / 'nnUNetTrainer_10epochs.py'
trainer_path.write_text(trainer_code)

from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_10epochs \
    import nnUNetTrainer_10epochs
print(f'✓ Trainer instalado: max_num_epochs = {nnUNetTrainer_10epochs.max_num_epochs}')

✓ Trainer instalado: max_num_epochs = 10


# Función de benchmark

In [7]:
import json, time, gc, torch, subprocess, shutil, os, signal, threading
import pandas as pd
import re
from pathlib import Path

def limpiar_resultados():
    results_path = Path(NNUNET_RESULTS) / DATASET_NAME
    if results_path.exists():
        shutil.rmtree(str(results_path))
    print('✓ Resultados anteriores eliminados')

def set_config(batch_size, workers):
    os.environ['nnUNet_n_proc_DA'] = str(workers)
    plans_path = Path(NNUNET_PREPROC) / DATASET_NAME / 'nnUNetPlans.json'
    with open(plans_path) as f:
        plans = json.load(f)
    plans['configurations']['3d_lowres']['batch_size'] = batch_size
    with open(plans_path, 'w') as f:
        json.dump(plans, f, indent=2)
    print(f'  batch_size={batch_size}, workers={workers}')

def limpiar_gpu():
    subprocess.run(['pkill', '-9', '-f', 'nnUNetv2_train'], capture_output=True)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    subprocess.run(['nvidia-smi', '--gpu-reset'], capture_output=True)
    time.sleep(3)

def contar_epochs_en_log():
    logs = sorted(Path(NNUNET_RESULTS).rglob('training_log*.txt'))
    if not logs:
        return 0
    try:
        return logs[-1].read_text().count('Epoch time:')
    except:
        return 0

def run_benchmark(batch_size, workers):
    print(f'\n{"="*55}')
    print(f'  batch={batch_size} | workers={workers}')
    print(f'{"="*55}')

    limpiar_resultados()
    set_config(batch_size, workers)
    limpiar_gpu()

    start            = time.time()
    epoch_times_live = []
    killed           = [False]
    process          = None

    cmd = [
        'nnUNetv2_train',
        str(DATASET_ID), CONFIG, '0',
        '-tr', TRAINER,
        '--npz',
    ]

    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        # Hilo monitor — lee log en disco cada 10s
        def monitor_log(proc, killed_flag):
            while not killed_flag[0]:
                epochs_done = contar_epochs_en_log()
                if epochs_done >= 10:
                    if not killed_flag[0]:  # ← evitar doble disparo
                        killed_flag[0] = True
                        print(f'\n  ✓ Monitor: {epochs_done} epochs completadas — matando proceso')
                        try:
                            os.kill(proc.pid, signal.SIGKILL)
                        except:
                            proc.kill()
                        subprocess.run(['pkill', '-9', '-f', 'nnUNetv2_train'], capture_output=True)
                    return
                time.sleep(10)

        monitor = threading.Thread(
            target=monitor_log,
            args=(process, killed),
            daemon=True
        )
        monitor.start()

        timeout_seconds = 5400

        for line in iter(process.stdout.readline, ''):
            if killed[0]:
                break

            if time.time() - start > timeout_seconds:
                killed[0] = True
                os.kill(process.pid, signal.SIGKILL)
                print(f'\n✗ TIMEOUT (>90 min)')
                limpiar_gpu()
                return None

            line = line.strip()

            if 'Epoch ' in line and 'time' not in line.lower() and 'best' not in line.lower():
                match = re.search(r'Epoch (\d+)', line)
                if match:
                    epoch_num = int(match.group(1))
                    elapsed   = time.time() - start
                    print(f'\n  → Epoch {epoch_num}/10 [{elapsed:.0f}s transcurridos]')

            elif 'Epoch time:' in line:
                match = re.search(r'Epoch time: ([\d.]+)', line)
                if match:
                    t          = float(match.group(1))
                    epoch_times_live.append(t)
                    avg_so_far = sum(epoch_times_live) / len(epoch_times_live)
                    remaining  = max(0, (10 - len(epoch_times_live)) * avg_so_far)
                    print(f'     Epoch time: {t:.1f}s | '
                          f'Promedio: {avg_so_far:.1f}s | '
                          f'Restante: {remaining:.0f}s')

            elif 'train_loss' in line:
                print(f'     {line}')
            elif 'val_loss' in line:
                print(f'     {line}')
            elif 'EMA pseudo Dice' in line:
                print(f'     {line}')
            elif 'OutOfMemoryError' in line or 'CUDA out of memory' in line:
                print(f'\n  ✗ OOM')
                killed[0] = True
                os.kill(process.pid, signal.SIGKILL)
                limpiar_gpu()
                return None
            elif 'Error' in line or 'Exception' in line:
                print(f'  ⚠ {line}')

        killed[0] = True
        # Forzar terminación sin esperar
        try:
            os.kill(process.pid, signal.SIGKILL)
        except:
            pass
        try:
            process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            pass
        # Limpiar procesos huérfanos
        subprocess.run(['pkill', '-9', '-f', 'nnUNetv2_train'], capture_output=True)
        limpiar_gpu()

    except Exception as e:
        killed[0] = True
        try:
            os.kill(process.pid, signal.SIGKILL)
        except:
            pass
        subprocess.run(['pkill', '-9', '-f', 'nnUNetv2_train'], capture_output=True)
        print(f'\n✗ Error inesperado: {e}')
        limpiar_gpu()
        return None

    # Leer tiempos del log si el buffer no los capturó todos
    logs = sorted(Path(NNUNET_RESULTS).rglob('training_log*.txt'))
    if logs and len(epoch_times_live) < 2:
        content = logs[-1].read_text()
        epoch_times_live = [float(m.group(1))
                           for m in re.finditer(r'Epoch time: ([\d.]+)', content)]

    epoch_times_clean = epoch_times_live[1:] if len(epoch_times_live) > 1 \
                        else epoch_times_live

    if not epoch_times_clean:
        print('⚠ No se encontraron tiempos de epoch')
        return None

    avg_epoch  = sum(epoch_times_clean) / len(epoch_times_clean)
    min_epoch  = min(epoch_times_clean)
    max_epoch  = max(epoch_times_clean)
    proj_fold  = avg_epoch * 250 / 3600
    proj_total = proj_fold * 5

    print(f'\nCompletado en {(time.time()-start)/60:.1f} min')
    print(f'  Epoch time promedio: {avg_epoch:.1f}s')
    print(f'  Min: {min_epoch:.1f}s | Max: {max_epoch:.1f}s')
    print(f'  Proyección fold (250 epochs): {proj_fold:.1f}h')
    print(f'  Proyección total (5 folds):   {proj_total:.1f}h')

    limpiar_gpu()

    return {
        'batch_size':   batch_size,
        'workers':      workers,
        'avg_epoch_s':  round(avg_epoch, 1),
        'min_epoch_s':  round(min_epoch, 1),
        'max_epoch_s':  round(max_epoch, 1),
        'proj_fold_h':  round(proj_fold, 2),
        'proj_total_h': round(proj_total, 2),
        'epoch_times':  epoch_times_live,
        'status':       'OK'
    }

print('✓ run_benchmark lista con hilo monitor')

✓ run_benchmark lista con hilo monitor


# Limpiamos GPU antes de entrenar

In [8]:
import torch
import gc
import subprocess
import time

# Limpiar memoria GPU
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Reset GPU
subprocess.run(['nvidia-smi', '--gpu-reset'], capture_output=True)
time.sleep(3)

# Verificar
free, total = torch.cuda.mem_get_info()
print(f'VRAM libre: {free/1e9:.1f} GB')
print(f'VRAM total: {total/1e9:.1f} GB')

VRAM libre: 23.5 GB
VRAM total: 23.7 GB


# Ejecutar benchmark nnUn-Net

In [9]:
TRAINER = 'nnUNetTrainer_10epochs'

CONFIGS_TO_TEST = [
    {'batch_size': 2, 'workers': 2},
    {'batch_size': 2, 'workers': 4},
    {'batch_size': 2, 'workers': 8},
    {'batch_size': 2, 'workers': 16},
    {'batch_size': 4, 'workers': 8},
    {'batch_size': 4, 'workers': 16},
    {'batch_size': 8, 'workers': 8},
    {'batch_size': 8, 'workers': 16},
]

resultados  = []
fallidos    = []
total       = len(CONFIGS_TO_TEST)

for i, cfg in enumerate(CONFIGS_TO_TEST):
    print(f'\n[{i+1}/{total}] Probando batch={cfg["batch_size"]} workers={cfg["workers"]}')

    r = run_benchmark(cfg['batch_size'], cfg['workers'])

    if r:
        resultados.append(r)
        # Guardar resultados parciales en Drive después de cada prueba exitosa
        df_parcial = pd.DataFrame(resultados)
        drive_results = Path('/content/drive/MyDrive/VerSe_2020_Dataset/3Dlowres_benchmark_results_nnUNet_v2')
        drive_results.mkdir(parents=True, exist_ok=True)
        df_parcial.to_csv(
            str(drive_results / 'benchmark_3d_lowres_nnunet_parcial.csv'),
            index=False
        )
        print(f'  ✓ Resultado guardado en Drive ({len(resultados)} pruebas hasta ahora)')
    else:
        fallidos.append({
            'batch_size': cfg['batch_size'],
            'workers':    cfg['workers'],
            'status':     'FALLIDO'
        })
        print(f'  ⚠ Combinación batch={cfg["batch_size"]} workers={cfg["workers"]} omitida')

print(f'\n{"="*55}')
print(f'BENCHMARK COMPLETADO')
print(f'  Exitosas: {len(resultados)}/{total}')
print(f'  Fallidas: {len(fallidos)}/{total}')
if fallidos:
    print(f'  Combinaciones fallidas:')
    for f in fallidos:
        print(f'    batch={f["batch_size"]} workers={f["workers"]}')
print(f'{"="*55}')


[1/8] Probando batch=2 workers=2

  batch=2 | workers=2
✓ Resultados anteriores eliminados
  batch_size=2, workers=2

  → Epoch 0/10 [1680s transcurridos]
     2026-05-03 21:50:42.187368: train_loss 0.3212
     2026-05-03 21:50:42.187649: val_loss 0.121
     Epoch time: 261.6s | Promedio: 261.6s | Restante: 2354s
     2026-05-03 21:50:42.188290: Yayy! New best EMA pseudo Dice: 0.0

  → Epoch 1/10 [1680s transcurridos]
     2026-05-03 21:54:07.280492: train_loss 0.1131
     2026-05-03 21:54:07.280816: val_loss 0.1086
     Epoch time: 203.7s | Promedio: 232.7s | Restante: 1861s

  → Epoch 2/10 [1680s transcurridos]
     2026-05-03 21:57:26.477160: train_loss 0.0963
     2026-05-03 21:57:26.477463: val_loss 0.0988
     Epoch time: 197.9s | Promedio: 221.1s | Restante: 1548s

  → Epoch 3/10 [1680s transcurridos]
     2026-05-03 22:00:48.131904: train_loss 0.096
     2026-05-03 22:00:48.132200: val_loss 0.138
     Epoch time: 200.4s | Promedio: 215.9s | Restante: 1295s

  → Epoch 4/10 [168

In [11]:
from pathlib import Path

# Buscar splits_final.json en todo el preprocessing
base = Path('/content/nnunet_verse')
splits = list(base.rglob('splits_final.json'))
for s in splits:
    print(s)

/content/nnunet_verse/nnUNet_preprocessed/Dataset507_VerSe2020/splits_final.json


# Resultados y tabla final nnU-Net

In [10]:
if resultados:
    df = pd.DataFrame(resultados)
    df = df.sort_values('avg_epoch_s')

    print('\n=== RESULTADOS nnU-Net 3D lowres — L4 ===\n')
    print(df[['batch_size', 'workers', 'avg_epoch_s',
              'min_epoch_s', 'max_epoch_s',
              'proj_fold_h', 'proj_total_h']].to_string(index=False))

    mejor = df.iloc[0]
    print(f'\n✓ MEJOR CONFIGURACIÓN:')
    print(f'  batch_size = {int(mejor["batch_size"])}')
    print(f'  workers    = {int(mejor["workers"])}')
    print(f'  Epoch time = {mejor["avg_epoch_s"]}s promedio')
    print(f'  Fold 250 epochs = {mejor["proj_fold_h"]}h')
    print(f'  Total 5 folds   = {mejor["proj_total_h"]}h')

    # Guardar en Drive
    drive_results = Path('/content/drive/MyDrive/VerSe_2020_Dataset/3Dlowres_benchmark_results_nnUNet_v2')
    drive_results.mkdir(parents=True, exist_ok=True)
    df.to_csv(str(drive_results / 'benchmark_3d_lowres_nnunet.csv'), index=False)
    print(f'\n✓ Resultados guardados en Drive')
else:
    print('No hay resultados que mostrar')


=== RESULTADOS nnU-Net 3D lowres — L4 ===

 batch_size  workers  avg_epoch_s  min_epoch_s  max_epoch_s  proj_fold_h  proj_total_h
          2       16        106.5        106.4        106.7         7.40         36.98
          2        8        106.9        106.8        106.9         7.42         37.11
          2        4        113.6        111.8        115.3         7.89         39.45
          2        2        197.9        179.3        203.7        13.74         68.70
          4       16        209.7        209.7        209.8        14.56         72.82
          4        8        210.2        210.1        210.3        14.60         72.98

✓ MEJOR CONFIGURACIÓN:
  batch_size = 2
  workers    = 16
  Epoch time = 106.5s promedio
  Fold 250 epochs = 7.4h
  Total 5 folds   = 36.98h

✓ Resultados guardados en Drive


# Instalamos MedNext porque no esta nativo

In [10]:
!pip install git+https://github.com/MIC-DKFZ/MedNeXt.git -q

import subprocess
result = subprocess.run(
    ['python', '-c', 'from nnunet_mednext import create_mednext_v1; import inspect; print(inspect.signature(create_mednext_v1))'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr[:300])

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 105.0 MB/s eta 0:00:00
(num_input_channels, num_classes, model_id, kernel_size=3, deep_supervision=False)




# Instalar trainer MedNeXt 10 epochs

In [11]:
import nnunetv2
from pathlib import Path

nnunet_dir  = Path(nnunetv2.__file__).parent
trainer_dir = nnunet_dir / 'training' / 'nnUNetTrainer' / 'variants' / 'network_architecture'
trainer_dir.mkdir(parents=True, exist_ok=True)

trainer_code = '''
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunet_mednext import create_mednext_v1
import torch

class MedNeXtWrapper(torch.nn.Module):
    """Wrapper para hacer MedNeXt compatible con nnU-Net v2"""
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.encoder = model
        self.decoder = model

    def forward(self, x):
        return self.model(x)

class nnUNetTrainerMedNeXt_10epochs(nnUNetTrainer):
    """MedNeXt v1 en nnU-Net v2 — pipeline intacto"""
    max_num_epochs = 10

    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.initial_lr = 1e-3  # ← después del super().__init__ para sobreescribir 0.01

    def build_network_architecture(self, architecture_class_name, arch_init_kwargs,
                                   arch_init_kwargs_req_import, num_input_channels,
                                   num_output_channels, enable_deep_supervision):
        base_model = create_mednext_v1(
            num_input_channels=num_input_channels,
            num_classes=num_output_channels,
            model_id='B',
            kernel_size=3,
            deep_supervision=enable_deep_supervision
        )
        return MedNeXtWrapper(base_model)
'''

trainer_path = trainer_dir / 'nnUNetTrainerMedNeXt_10epochs.py'
trainer_path.write_text(trainer_code)

import subprocess
result = subprocess.run(
    ['python', '-c',
     'from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerMedNeXt_10epochs import nnUNetTrainerMedNeXt_10epochs; print(f"✓ max_epochs={nnUNetTrainerMedNeXt_10epochs.max_num_epochs}"); t = nnUNetTrainerMedNeXt_10epochs.__init__; print("✓ __init__ sobreescrito")'],
    capture_output=True, text=True
)
print(result.stdout if result.returncode == 0 else f'✗ {result.stderr}')

✓ max_epochs=10
✓ __init__ sobreescrito



In [12]:
import subprocess
result = subprocess.run(
    ['python', '-c', '''
from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerMedNeXt_10epochs import nnUNetTrainerMedNeXt_10epochs
print(f"initial_lr = {nnUNetTrainerMedNeXt_10epochs.initial_lr}")
print(f"max_num_epochs = {nnUNetTrainerMedNeXt_10epochs.max_num_epochs}")
'''],
    capture_output=True, text=True
)
print(result.stdout if result.returncode == 0 else result.stderr)

Traceback (most recent call last):
  File "<string>", line 3, in <module>
AttributeError: type object 'nnUNetTrainerMedNeXt_10epochs' has no attribute 'initial_lr'. Did you mean: 'initialize'?



In [13]:
import subprocess
result = subprocess.run(
    ['python', '-c', '''
import nnunetv2, inspect
from pathlib import Path

trainer_file = Path(nnunetv2.__file__).parent / "training" / "nnUNetTrainer" / "nnUNetTrainer.py"
lines = trainer_file.read_text().split("\\n")
for i, line in enumerate(lines):
    if "initial_lr" in line or "configure_optimizers" in line or "self.optimizer" in line.lower():
        print(f"{i+1}: {line}")
'''],
    capture_output=True, text=True
)
print(result.stdout[:3000])

152:         self.initial_lr = 1e-2
169:         self.optimizer = self.lr_scheduler = None  # -> self.initialize
248:             self.optimizer, self.lr_scheduler = self.configure_optimizers()
264:                 "initial_lr": self.initial_lr,
551:     def configure_optimizers(self):
552:         optimizer = torch.optim.SGD(self.network.parameters(), self.initial_lr, weight_decay=self.weight_decay,
554:         lr_scheduler = PolyLRScheduler(optimizer, self.initial_lr, self.num_epochs)
573:                 # self.optimizer.zero_grad()
1014:             f"Current learning rate: {np.round(self.optimizer.param_groups[0]['lr'], decimals=5)}")
1016:         self.logger.log('lrs', self.optimizer.param_groups[0]['lr'], self.current_epoch)
1028:         self.optimizer.zero_grad(set_to_none=True)
1040:             self.grad_scaler.unscale_(self.optimizer)
1042:             self.grad_scaler.step(self.optimizer)
1047:             self.optimizer.step()
1206:                     'optimizer_state'

# Benchmark MedNeXt

In [17]:
# Actualizar trainer para MedNeXt
TRAINER = 'nnUNetTrainerMedNeXt_10epochs'

# Solo probar batch=2 con workers=4 y 8
# batch=4+ causará OOM igual que nnU-Net
CONFIGS_MEDNEXT = [
    {'batch_size': 2, 'workers': 2},
    {'batch_size': 2, 'workers': 4},
    {'batch_size': 2, 'workers': 8},
    {'batch_size': 2, 'workers': 16},
    {'batch_size': 4, 'workers': 8},
    {'batch_size': 4, 'workers': 16},
    {'batch_size': 8, 'workers': 8},
    {'batch_size': 8, 'workers': 16},
]

resultados_mednext = []
fallidos_mednext   = []
total = len(CONFIGS_MEDNEXT)

for i, cfg in enumerate(CONFIGS_MEDNEXT):
    print(f'\n[{i+1}/{total}] MedNeXt — batch={cfg["batch_size"]} workers={cfg["workers"]}')
    r = run_benchmark(cfg['batch_size'], cfg['workers'])
    if r:
        r['modelo'] = 'MedNeXt'
        resultados_mednext.append(r)
        df_parcial = pd.DataFrame(resultados_mednext)
        drive_results = Path('/content/drive/MyDrive/VerSe_2020_Dataset/3Dlowres_benchmark_results_MedNeXt')
        drive_results.mkdir(parents=True, exist_ok=True)
        df_parcial.to_csv(
            str(drive_results / 'benchmark_3d_lowres_mednext_parcial.csv'),
            index=False
        )
        print(f'  ✓ Guardado en Drive ({len(resultados_mednext)} pruebas)')
    else:
        fallidos_mednext.append(cfg)
        print(f'  ⚠ Combinación omitida')

print(f'\n{"="*55}')
print(f'BENCHMARK MEDNEXT COMPLETADO')
print(f'  Exitosas: {len(resultados_mednext)}/{total}')
print(f'  Fallidas: {len(fallidos_mednext)}/{total}')
print(f'{"="*55}')


[1/3] MedNeXt — batch=2 workers=4

  batch=2 | workers=4
✓ Resultados anteriores eliminados
  batch_size=2, workers=4

  → Epoch 0/10 [1898s transcurridos]
     2026-05-03 03:31:27.720952: train_loss 0.5443
     2026-05-03 03:31:27.721380: val_loss 0.2182
     Epoch time: 284.4s | Promedio: 284.4s | Restante: 2559s
     2026-05-03 03:31:27.722064: Yayy! New best EMA pseudo Dice: 0.0034000000450760126

  → Epoch 1/10 [1898s transcurridos]
     2026-05-03 03:35:54.819851: train_loss 0.1601
     2026-05-03 03:35:54.820147: val_loss 0.1337
     Epoch time: 265.6s | Promedio: 275.0s | Restante: 2200s

  → Epoch 2/10 [1898s transcurridos]
     2026-05-03 03:40:21.898745: train_loss 0.1143
     2026-05-03 03:40:21.899066: val_loss 0.1315
     Epoch time: 265.8s | Promedio: 271.9s | Restante: 1903s

  → Epoch 3/10 [1898s transcurridos]
     2026-05-03 03:44:49.118892: train_loss 0.1021
     2026-05-03 03:44:49.119213: val_loss 0.1071
     Epoch time: 265.9s | Promedio: 270.4s | Restante: 1623

# Resultados y tabla final MedNeXt

In [18]:
if resultados_mednext:
    df_m = pd.DataFrame(resultados_mednext)
    df_m = df_m.sort_values('avg_epoch_s')
    print('\n=== RESULTADOS MedNeXt 3D lowres — L4 ===\n')
    print(df_m[['batch_size', 'workers', 'avg_epoch_s',
                'min_epoch_s', 'max_epoch_s',
                'proj_fold_h', 'proj_total_h']].to_string(index=False))
    mejor_m = df_m.iloc[0]
    print(f'\n✓ MEJOR CONFIGURACIÓN MedNeXt:')
    print(f'  batch_size = {int(mejor_m["batch_size"])}')
    print(f'  workers    = {int(mejor_m["workers"])}')
    print(f'  Epoch time = {mejor_m["avg_epoch_s"]}s')
    print(f'  Fold 250 epochs = {mejor_m["proj_fold_h"]}h')
    print(f'  Total 5 folds   = {mejor_m["proj_total_h"]}h')

    # Guardar final
    df_m.to_csv(
        str(drive_results / 'benchmark_3d_lowres_mednext_final.csv'),
        index=False
    )
    print(f'\n✓ Resultados finales guardados en Drive')


=== RESULTADOS MedNeXt 3D lowres — L4 ===

 batch_size  workers  avg_epoch_s  min_epoch_s  max_epoch_s  proj_fold_h  proj_total_h
          2       16        265.1        265.1        265.1        18.41         92.05
          2        4        265.8        265.6        265.9        18.46         92.30
          2        8        266.2        266.2        266.2        18.49         92.43

✓ MEJOR CONFIGURACIÓN MedNeXt:
  batch_size = 2
  workers    = 16
  Epoch time = 265.1s
  Fold 250 epochs = 18.41h
  Total 5 folds   = 92.05h

✓ Resultados finales guardados en Drive


# Instalamos Swin UNETR

In [19]:
!pip install monai -q

import subprocess
result = subprocess.run(
    ['python', '-c', 'from monai.networks.nets import SwinUNETR; print("✓ SwinUNETR disponible")'],
    capture_output=True, text=True
)
print(result.stdout if result.returncode == 0 else f'✗ {result.stderr}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 102.5 MB/s eta 0:00:00
✓ SwinUNETR disponible



In [22]:
import subprocess
result = subprocess.run(
    ['python', '-c', 'from monai.networks.nets import SwinUNETR; import inspect; print(inspect.signature(SwinUNETR.__init__))'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr[:300])

(self, in_channels: 'int', out_channels: 'int', patch_size: 'int' = 2, depths: 'Sequence[int]' = (2, 2, 2, 2), num_heads: 'Sequence[int]' = (3, 6, 12, 24), window_size: 'Sequence[int] | int' = 7, qkv_bias: 'bool' = True, mlp_ratio: 'float' = 4.0, feature_size: 'int' = 24, norm_name: 'tuple | str' = 'instance', drop_rate: 'float' = 0.0, attn_drop_rate: 'float' = 0.0, dropout_path_rate: 'float' = 0.0, normalize: 'bool' = True, norm_layer: 'type[LayerNorm]' = <class 'torch.nn.modules.normalization.LayerNorm'>, patch_norm: 'bool' = False, use_checkpoint: 'bool' = False, spatial_dims: 'int' = 3, downsample: 'str | nn.Module' = 'merging', use_v2: 'bool' = False) -> 'None'

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-05-03 05:49:14.866262: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. Yo


# Instalar trainer Swin UNETR 10 epochs

In [25]:
import nnunetv2
from pathlib import Path

nnunet_dir  = Path(nnunetv2.__file__).parent
trainer_dir = nnunet_dir / 'training' / 'nnUNetTrainer' / 'variants' / 'network_architecture'
trainer_dir.mkdir(parents=True, exist_ok=True)

trainer_code = '''
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from monai.networks.nets import SwinUNETR
import torch

class SwinUNETRWrapper(torch.nn.Module):
    """Wrapper para hacer Swin UNETR compatible con nnU-Net v2"""
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.encoder = model
        self.decoder = model

    def forward(self, x):
        output = self.model(x)
        # nnU-Net espera una lista de outputs (deep supervision)
        # Swin UNETR devuelve un solo tensor — lo envolvemos en lista
        if not isinstance(output, (list, tuple)):
            output = [output]
        return output

class nnUNetTrainerSwinUNETR_10epochs(nnUNetTrainer):
    """Swin UNETR en nnU-Net v2 — pipeline intacto"""
    max_num_epochs = 10

    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.initial_lr = 1e-3

    def build_network_architecture(self, architecture_class_name, arch_init_kwargs,
                                   arch_init_kwargs_req_import, num_input_channels,
                                   num_output_channels, enable_deep_supervision):
        base_model = SwinUNETR(
            in_channels=num_input_channels,
            out_channels=num_output_channels,
            feature_size=48,
            use_checkpoint=True,
            spatial_dims=3
        )
        return SwinUNETRWrapper(base_model)

    def set_deep_supervision_enabled(self, enabled: bool):
        pass
'''

trainer_path = trainer_dir / 'nnUNetTrainerSwinUNETR_10epochs.py'
trainer_path.write_text(trainer_code)

import subprocess
result = subprocess.run(
    ['python', '-c',
     'from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerSwinUNETR_10epochs import nnUNetTrainerSwinUNETR_10epochs; print(f"✓ max_epochs={nnUNetTrainerSwinUNETR_10epochs.max_num_epochs}")'],
    capture_output=True, text=True
)
print(result.stdout if result.returncode == 0 else f'✗ {result.stderr}')

✓ max_epochs=10



# Benchmark Swin UNETR

In [26]:
TRAINER = 'nnUNetTrainerSwinUNETR_10epochs'

CONFIGS_SWIN = [
    {'batch_size': 2, 'workers': 2},
    {'batch_size': 2, 'workers': 4},
    {'batch_size': 2, 'workers': 8},
    {'batch_size': 2, 'workers': 16},
    {'batch_size': 4, 'workers': 8},
    {'batch_size': 4, 'workers': 16},
    {'batch_size': 8, 'workers': 8},
    {'batch_size': 8, 'workers': 16},
]

resultados_swin = []
fallidos_swin   = []
total = len(CONFIGS_SWIN)

for i, cfg in enumerate(CONFIGS_SWIN):
    print(f'\n[{i+1}/{total}] Swin UNETR — batch={cfg["batch_size"]} workers={cfg["workers"]}')
    r = run_benchmark(cfg['batch_size'], cfg['workers'])
    if r:
        r['modelo'] = 'SwinUNETR'
        resultados_swin.append(r)
        df_parcial = pd.DataFrame(resultados_swin)
        drive_results = Path('/content/drive/MyDrive/VerSe_2020_Dataset/3Dlowres_benchmark_results_swinunetr')
        drive_results.mkdir(parents=True, exist_ok=True)
        df_parcial.to_csv(
            str(drive_results / 'benchmark_3d_lowres_swinunetr_parcial.csv'),
            index=False
        )
        print(f'  ✓ Guardado en Drive ({len(resultados_swin)} pruebas)')
    else:
        fallidos_swin.append(cfg)
        print(f'  ⚠ Combinación omitida')

print(f'\n{"="*55}')
print(f'BENCHMARK SWIN UNETR COMPLETADO')
print(f'  Exitosas: {len(resultados_swin)}/{total}')
print(f'  Fallidas: {len(fallidos_swin)}/{total}')
print(f'{"="*55}')


[1/8] Swin UNETR — batch=2 workers=2

  batch=2 | workers=2
✓ Resultados anteriores eliminados
  batch_size=2, workers=2

  → Epoch 0/10 [2404s transcurridos]
     2026-05-03 06:07:31.983565: train_loss 0.2273
     2026-05-03 06:07:31.983853: val_loss 0.072
     Epoch time: 567.6s | Promedio: 567.6s | Restante: 5109s
     2026-05-03 06:07:31.984632: Yayy! New best EMA pseudo Dice: 0.0

  → Epoch 1/10 [2404s transcurridos]
     2026-05-03 06:12:34.171650: train_loss 0.0592
     2026-05-03 06:12:34.171941: val_loss 0.0596
     Epoch time: 300.4s | Promedio: 434.0s | Restante: 3472s

  → Epoch 2/10 [2404s transcurridos]
     2026-05-03 06:17:36.311602: train_loss 0.0531
     2026-05-03 06:17:36.311901: val_loss 0.0512
     Epoch time: 300.9s | Promedio: 389.6s | Restante: 2727s

  → Epoch 3/10 [2404s transcurridos]
     2026-05-03 06:22:38.695456: train_loss 0.0498
     2026-05-03 06:22:38.695763: val_loss 0.0489
     Epoch time: 301.0s | Promedio: 367.5s | Restante: 2205s
     2026-05-0

# Resultados y tabla final Swin UNETR

In [27]:
if resultados_swin:
    df_s = pd.DataFrame(resultados_swin)
    df_s = df_s.sort_values('avg_epoch_s')
    print('\n=== RESULTADOS Swin UNETR 3D lowres — L4 ===\n')
    print(df_s[['batch_size', 'workers', 'avg_epoch_s',
                'min_epoch_s', 'max_epoch_s',
                'proj_fold_h', 'proj_total_h']].to_string(index=False))
    mejor_s = df_s.iloc[0]
    print(f'\n✓ MEJOR CONFIGURACIÓN Swin UNETR:')
    print(f'  batch_size = {int(mejor_s["batch_size"])}')
    print(f'  workers    = {int(mejor_s["workers"])}')
    print(f'  Epoch time = {mejor_s["avg_epoch_s"]}s')
    print(f'  Fold 250 epochs = {mejor_s["proj_fold_h"]}h')
    print(f'  Total 5 folds   = {mejor_s["proj_total_h"]}h')

    df_s.to_csv(
        str(drive_results / 'benchmark_3d_lowres_swinunetr_final.csv'),
        index=False
    )
    print(f'\n✓ Resultados finales guardados en Drive')


=== RESULTADOS Swin UNETR 3D lowres — L4 ===

 batch_size  workers  avg_epoch_s  min_epoch_s  max_epoch_s  proj_fold_h  proj_total_h
          2        4        300.7        300.2        301.2        20.88        104.41
          2        2        301.0        300.4        301.3        20.90        104.51
          2        8        302.1        301.7        302.5        20.98        104.88
          2       16        302.4        301.9        302.8        21.00        105.00

✓ MEJOR CONFIGURACIÓN Swin UNETR:
  batch_size = 2
  workers    = 4
  Epoch time = 300.7s
  Fold 250 epochs = 20.88h
  Total 5 folds   = 104.41h

✓ Resultados finales guardados en Drive
